In [1]:
# Cell 1 - imports
from pathlib import Path
import numpy as np
from ultralytics import YOLO

In [2]:
# Cell 2 - config
DATA_YAML = r"C:\Users\manis\Documents\WT\yolo_dataset\data.yaml"
MODEL     = r"C:\Users\Manish\Documents\WT\runs\detect\runs\detect\turbine_damage-4\weights\best.pt"
BEST_PT = r"C:\Users\manis\Documents\WT\runs\detect\runs\detect\turbine_damage-4\weights\best.pt"
EPOCHS    = 100
IMGSZ     = 640
BATCH     = 4
DEVICE    = 'cpu'


In [3]:
# Cell 4 - helper: classification accuracy from the validation confusion matrix
def classification_from_confusion_matrix(metrics, names):
    """
    Ultralytics' confusion matrix has shape (nc+1, nc+1): rows = predicted, cols = true,
    last index = background.
      - Classification accuracy (matched boxes only): of the detections that were matched
        to a ground-truth box, how many got the right class.
      - Detection-aware accuracy: also counts missed objects (FN) and false alarms (FP) as errors.
    """
    cm = np.asarray(metrics.confusion_matrix.matrix)
    nc = len(names)
    core = cm[:nc, :nc]                       # class-vs-class block (no background)

    correct = np.trace(core)
    matched = core.sum()
    cls_acc = correct / matched if matched else float("nan")
    det_aware_acc = correct / cm.sum() if cm.sum() else float("nan")

    print(f"\nClassification accuracy (matched detections): {cls_acc:.4f}  ({int(correct)}/{int(matched)})")
    print(f"Detection-aware accuracy (incl. FN + FP):     {det_aware_acc:.4f}")

    print("\nPer-class classification accuracy (matched boxes):")
    for i in range(nc):
        col = core[:, i].sum()                # all matched boxes whose true class is i
        acc = core[i, i] / col if col else float("nan")
        print(f"  {names[i]:<20s} {acc:.4f}  ({int(core[i, i])}/{int(col)})")
    return cls_acc, det_aware_acc


In [4]:
# Cell 5 - validate best checkpoint (detection metrics + classification accuracy)
best_model = YOLO(BEST_PT)

metrics = best_model.val(data=DATA_YAML, split="val", imgsz=IMGSZ, device=DEVICE, plots=True)

print("\n=== Object detection metrics ===")
print(f"mAP50-95  : {metrics.box.map:.4f}")
print(f"mAP50     : {metrics.box.map50:.4f}")
print(f"mAP75     : {metrics.box.map75:.4f}")
print(f"Precision : {metrics.box.mp:.4f}")
print(f"Recall    : {metrics.box.mr:.4f}")

print("\nPer-class (P / R / AP50 / AP50-95):")
for k, c in enumerate(metrics.box.ap_class_index):
    print(f"  {metrics.names[c]:<20s} "
          f"{metrics.box.p[k]:.3f}  {metrics.box.r[k]:.3f}  "
          f"{metrics.box.ap50[k]:.3f}  {metrics.box.ap[k]:.3f}")

print("\n=== Classification ===")
cls_acc, det_acc = classification_from_confusion_matrix(metrics, metrics.names)


Ultralytics 8.4.174  Python-3.14.7 torch-2.14.1+cpu CPU (Intel Core Ultra 7 270HX Plus)
YOLO26n summary (fused): 120 layers, 2,376,006 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 573.723.8 MB/s, size: 50.8 KB)
val: Scanning C:\Users\manis\Documents\WT\yolo_dataset\labels\val.cache... 213 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 213/213 59.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 2.6it/s 5.4s0.4s
                   all        213        316      0.832      0.751      0.831      0.469
             corrosion         35         56      0.749        0.5      0.636      0.293
                 crack         35         41      0.895      0.829      0.915      0.452
                 craze         36         45      0.849      0.822      0.859      0.502
            hide_craze         53         71      0.749      0.676      0.732      0.385
        surface_inju

In [5]:
# Cell 6 - standalone inference + per-object classification report
import yaml
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
from ultralytics import YOLO

CONF_THRES  = 0.25
IOU_MATCH   = 0.5          # IoU needed to match a prediction to a ground-truth box

root       = Path(DATA_YAML).parent
img_dir    = root / "images" / "val"
lbl_dir    = root / "labels" / "val"
class_names = yaml.safe_load(open(DATA_YAML))["names"]
if isinstance(class_names, dict):
    class_names = [class_names[i] for i in sorted(class_names)]
BG = len(class_names)      # index used for "background / missed / false alarm"

def iou_matrix(a, b):
    """a: (N,4), b: (M,4) in xyxy -> (N,M) IoU"""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)

def load_gt(lbl_path, w, h):
    boxes, cls = [], []
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            c, x, y, bw, bh = line.split()[:5]
            x, y, bw, bh = float(x) * w, float(y) * h, float(bw) * w, float(bh) * h
            boxes.append([x - bw / 2, y - bh / 2, x + bw / 2, y + bh / 2]); cls.append(int(c))
    return np.array(boxes).reshape(-1, 4), np.array(cls, dtype=int)

y_true, y_pred = [], []        # includes BG for misses / false alarms
matched_true, matched_pred = [], []   # matched boxes only (pure classification)

img_files = sorted(p for p in img_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
preds = best_model.predict(source=str(img_dir), conf=CONF_THRES, imgsz=IMGSZ,
                           device=DEVICE, save=True, stream=True, verbose=False)

for img_path, r in zip(img_files, preds):
    h, w = r.orig_shape
    gt_boxes, gt_cls = load_gt(lbl_dir / f"{img_path.stem}.txt", w, h)
    pr_boxes = r.boxes.xyxy.cpu().numpy()
    pr_cls   = r.boxes.cls.cpu().numpy().astype(int)
    pr_conf  = r.boxes.conf.cpu().numpy()

    order = np.argsort(-pr_conf)                 # greedy match, highest confidence first
    pr_boxes, pr_cls = pr_boxes[order], pr_cls[order]
    ious = iou_matrix(gt_boxes, pr_boxes)
    gt_used, pr_used = set(), set()
    pairs = sorted(((ious[g, p], g, p) for g in range(len(gt_boxes)) for p in range(len(pr_boxes))
                    if ious[g, p] >= IOU_MATCH), reverse=True)
    for _, g, p in pairs:
        if g in gt_used or p in pr_used:
            continue
        gt_used.add(g); pr_used.add(p)
        y_true.append(gt_cls[g]); y_pred.append(pr_cls[p])
        matched_true.append(gt_cls[g]); matched_pred.append(pr_cls[p])
    for g in range(len(gt_boxes)):               # missed objects
        if g not in gt_used:
            y_true.append(gt_cls[g]); y_pred.append(BG)
    for p in range(len(pr_boxes)):               # false alarms
        if p not in pr_used:
            y_true.append(BG); y_pred.append(pr_cls[p])

labels_all = list(range(len(class_names)))
print(f"\nMatched objects: {len(matched_true)}")
print(f"Classification accuracy (matched boxes): {accuracy_score(matched_true, matched_pred):.4f}")
print(f"Detection-aware accuracy (incl. misses/false alarms): {accuracy_score(y_true, y_pred):.4f}")

print("\nPer-class report (matched boxes):")
print(classification_report(matched_true, matched_pred, labels=labels_all,
                            target_names=class_names, zero_division=0))

print("Confusion matrix (rows = true, cols = predicted; last = background):")
print(confusion_matrix(y_true, y_pred, labels=labels_all + [BG]))
print("\nAnnotated images saved to:", best_model.predictor.save_dir)



Matched objects: 230
Classification accuracy (matched boxes): 0.9870
Detection-aware accuracy (incl. misses/false alarms): 0.5470

Per-class report (matched boxes):
                precision    recall  f1-score   support

     corrosion       1.00      0.97      0.98        33
         crack       1.00      1.00      1.00        33
         craze       0.97      1.00      0.99        38
    hide_craze       0.96      1.00      0.98        48
surface_injure       1.00      0.98      0.99        61
 thunderstrike       1.00      0.94      0.97        17

      accuracy                           0.99       230
     macro avg       0.99      0.98      0.99       230
  weighted avg       0.99      0.99      0.99       230

Confusion matrix (rows = true, cols = predicted; last = background):
[[32  0  0  1  0  0 23]
 [ 0 33  0  0  0  0  8]
 [ 0  0 38  0  0  0  7]
 [ 0  0  0 48  0  0 23]
 [ 0  0  0  1 60  0 23]
 [ 0  0  1  0  0 16  2]
 [26 17 11 21 18  6  0]]

Annotated images saved to: C:\Us

In [5]:
# Cell 7 - image-level classification over the ENTIRE dataset, with progress notifications
import time, yaml
import numpy as np, pandas as pd
from collections import Counter
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix

CONF_THRES = 0.25          # lower (e.g. 0.1) if too many images get "no_detection"
root = Path(DATA_YAML).parent
class_names = yaml.safe_load(open(DATA_YAML))["names"]
if isinstance(class_names, dict):
    class_names = [class_names[i] for i in sorted(class_names)]
BG = len(class_names)
names_bg = class_names + ["no_detection"]

def notify(title, msg):
    """Beep + Windows toast (toast needs: pip install plyer; silently skipped if missing)."""
    print(f"\n🔔 {title}: {msg}")
    try:
        import winsound; winsound.Beep(1000, 400); winsound.Beep(1400, 400)
    except Exception:
        pass
    try:
        from plyer import notification
        notification.notify(title=title, message=msg, timeout=8)
    except Exception:
        pass

# ---- collect work list so we know the totals up front ----
splits = {}
for s in ["train", "val", "test"]:
    d = root / "images" / s
    if d.exists():
        splits[s] = sorted(p for p in d.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
total = sum(len(v) for v in splits.values())
print(f"Found {total} images across splits: " + ", ".join(f"{k}={len(v)}" for k, v in splits.items()))
notify("Classification started", f"{total} images, splits: {', '.join(splits)}")

rows, t_start = [], time.time()
done = 0
overall = tqdm(total=total, desc="OVERALL", position=0, unit="img")

for split, imgs in splits.items():
    lbl_dir = root / "labels" / split
    correct, n, t_split = 0, 0, time.time()
    bar = tqdm(total=len(imgs), desc=f"{split:<5}", position=1, leave=False, unit="img")
    for r in best_model.predict(source=[str(p) for p in imgs], conf=CONF_THRES, imgsz=IMGSZ,
                                device=DEVICE, stream=True, verbose=False, save=False):
        stem = Path(r.path).stem
        lbl = lbl_dir / f"{stem}.txt"
        gt = [int(l.split()[0]) for l in lbl.read_text().strip().splitlines()] if lbl.exists() else []
        gt_label = Counter(gt).most_common(1)[0][0] if gt else BG
        if len(r.boxes):
            k = int(r.boxes.conf.argmax()); pred, conf = int(r.boxes.cls[k]), float(r.boxes.conf[k])
        else:
            pred, conf = BG, 0.0
        ok = pred == gt_label
        rows.append(dict(split=split, image=Path(r.path).name,
                         gt_classes=sorted({names_bg[c] for c in gt}) or ["none"],
                         gt_label=names_bg[gt_label], pred_label=names_bg[pred], conf=round(conf, 3),
                         correct=ok, correct_any=(pred in set(gt)) if gt else (pred == BG),
                         multi_class_image=len(set(gt)) > 1))
        n += 1; correct += ok; done += 1
        bar.update(1); overall.update(1)
        bar.set_postfix(acc=f"{correct/n:.3f}")                       # running accuracy for this split
        overall.set_postfix(acc=f"{sum(x['correct'] for x in rows)/done:.3f}")
    bar.close()
    msg = f"{split}: {n} images, top1_acc={correct/n:.4f}, {time.time()-t_split:.0f}s"
    tqdm.write("✅ " + msg)
    notify(f"Split finished ({done}/{total})", msg) if split != list(splits)[-1] else None
overall.close()

# ---- results ----
df = pd.DataFrame(rows)
df.to_csv("image_level_predictions.csv", index=False)

print(f"\n{'split':<8}{'images':>8}{'top1_acc':>10}{'lenient_acc':>13}")
for s, d in df.groupby("split", sort=False):
    print(f"{s:<8}{len(d):>8}{d['correct'].mean():>10.4f}{d['correct_any'].mean():>13.4f}")
print(f"{'ALL':<8}{len(df):>8}{df['correct'].mean():>10.4f}{df['correct_any'].mean():>13.4f}")
print(f"\nImages with >1 GT class: {int(df['multi_class_image'].sum())}")

print("\nPer-class report, entire dataset (image-level):")
print(classification_report(df["gt_label"], df["pred_label"], labels=names_bg[:BG], zero_division=0))
print("Confusion matrix (rows = true, cols = predicted; last = no_detection):")
print(pd.DataFrame(confusion_matrix(df["gt_label"], df["pred_label"], labels=names_bg),
                   index=names_bg, columns=names_bg))

notify("Classification complete",
       f"{len(df)} images in {time.time()-t_start:.0f}s | top1_acc={df['correct'].mean():.4f} | saved image_level_predictions.csv")

Found 1065 images across splits: train=852, val=213

🔔 Classification started: 1065 images, splits: train, val


OVERALL:   0%|          | 0/1065 [00:00<?, ?img/s]

train:   0%|          | 0/852 [00:00<?, ?img/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



✅ val: 213 images, top1_acc=0.8826, 6s

split     images  top1_acc  lenient_acc
train        852    0.9272       0.9566
val          213    0.8826       0.9014
ALL         1065    0.9183       0.9455

Images with >1 GT class: 84

Per-class report, entire dataset (image-level):
                precision    recall  f1-score   support

     corrosion       0.99      0.90      0.94       169
         crack       1.00      0.98      0.99       177
         craze       0.94      0.96      0.95       182
    hide_craze       0.97      0.85      0.91       229
surface_injure       0.94      0.92      0.93       237
 thunderstrike       0.88      0.93      0.90        71

     micro avg       0.96      0.92      0.94      1065
     macro avg       0.95      0.92      0.94      1065
  weighted avg       0.96      0.92      0.94      1065

Confusion matrix (rows = true, cols = predicted; last = no_detection):
                corrosion  crack  craze  hide_craze  surface_injure  \
corrosion        